# EV3 — External validation on real frost-damaged vineyard images (Manisa, Turkey, April 2025)
Tests whether models trained on WEAPD (leakage-free v2 split) transfer to a real crop-field frost event.
Dataset: "Frost Damaged Leaf and Grape" (IEEE DataPort) via Roboflow Universe "Grape-Segmentation - yedek - kendi verim",
CC BY 4.0, 909 images, COCO segmentation masks (frozen-leaf, frozen-grape, healthy-leaf, healthy-grape).
Image label: **frost_affected = 1** if any frozen-* mask is present, else 0 (516 vs 393).

Experiments
1. **Zero-shot**: the 11-class WEAPD model on EV3. Does the cold-protection score (p(frost)+p(rime)+p(glaze)+p(snow))
   separate frost-damaged from healthy vines? Do conformal sets (calibrated on WEAPD val) grow under domain shift?
2. **Few-shot linear probe**: frozen features from the WEAPD-trained backbone vs the same backbone with ImageNet weights,
   k = 5…100 labelled images per class, 20 repeats, identical samples for both inits (paired Wilcoxon test).
3. **Binary action-level conformal + cost-sensitive threshold** on the probe (calibration split ≠ test split).
4. **Full fine-tuning** from WEAPD vs ImageNet initialisation, 3 seeds.

**Inputs to add:** (a) your private dataset `ev3-frost-grape` (the Roboflow COCO-segmentation zip);
(b) the WEAPD weather dataset used before; (c) the output of the **v2 seed-0 notebook** (contains
`results_v2/split.csv` and `best_<model>_s0.pt`). If (c) is missing the notebook trains the WEAPD model itself
(TRAIN_IF_MISSING). **Accelerator: GPU T4 x2.** Download `ev3_results_small.zip` from the Output tab.

In [ ]:
CFG = dict(
    BACKBONES=["swin_tiny_patch4_window7_224", "convnext_tiny"],
    COLD_CLASSES=["frost", "rime", "glaze", "snow"],          # cold_protection action (paper mapping)
    ACTION_MAP={"frost": "cold_protection", "rime": "cold_protection", "glaze": "cold_protection", "snow": "cold_protection",
                "hail": "hail_protection", "dew": "disease_alert", "fogsmog": "disease_alert", "rain": "disease_alert",
                "sandstorm": "wind_dust", "rainbow": "no_action", "lightning": "no_action"},
    ALPHAS=[0.05, 0.10],
    KSHOTS=[5, 10, 25, 50, 100], REPEATS=20,
    COST_RATIOS=[1, 2, 5, 10],                                 # cost of a missed frost-damaged plant vs a false alarm
    FT=True, FT_SEEDS=[0, 1, 2], FT_EPOCHS=12, FT_LR=1e-4, FT_BATCH=32,
    TRAIN_IF_MISSING=True, WEAPD_EPOCHS=15,
    EV3_SPLIT_SEED=42, PHASH_MAX_DIST=8,
    IMG_RESIZE=256, IMG_SIZE=224, NUM_WORKERS=2,
    OUT="/kaggle/working/ev3_results",
)
import os, glob, json, time, hashlib, random, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import timm, torchvision.transforms as TV
from PIL import Image, ImageFile, ImageDraw
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from scipy.stats import wilcoxon
ImageFile.LOAD_TRUNCATED_IMAGES = True; warnings.filterwarnings("ignore")
os.makedirs(CFG["OUT"], exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(CFG, "\ndevice:", DEVICE, torch.cuda.device_count(), "GPU(s)")
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# 1. Locate inputs
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
ann = [p for p in glob.glob("/kaggle/input/**/_annotations.coco.json", recursive=True)]
assert ann, "EV3 not found: add the ev3-frost-grape dataset as an Input."
EV3_ROOT = os.path.dirname(os.path.dirname(ann[0])); print("EV3_ROOT:", EV3_ROOT)
WEAPD_NAMES = {"dew", "fogsmog", "frost", "glaze", "hail", "lightning", "rain", "rainbow", "rime", "sandstorm", "snow"}
WEAPD_ROOT = None
for d, sub, _ in os.walk("/kaggle/input", followlinks=True):
    if len(WEAPD_NAMES & {s.lower() for s in sub}) >= 9: WEAPD_ROOT = d; break
print("WEAPD_ROOT:", WEAPD_ROOT)
SPLIT_CSV = next(iter(glob.glob("/kaggle/input/**/results_v2/split.csv", recursive=True)), None)
print("v2 split.csv:", SPLIT_CSV)
CKPT = {b: next(iter(glob.glob(f"/kaggle/input/**/best_{b}_s0.pt", recursive=True)), None) for b in CFG["BACKBONES"]}
print("checkpoints:", CKPT)

In [ ]:
# 2. EV3 image labels from COCO masks + near-duplicate groups + grouped stratified split (test / cal / pool)
rows = []
for s in ["train", "valid", "test"]:
    j = json.load(open(os.path.join(EV3_ROOT, s, "_annotations.coco.json"))); cats = {c["id"]: c["name"] for c in j["categories"]}
    per, area = {}, {}
    for a in j["annotations"]:
        n = cats[a["category_id"]]; per.setdefault(a["image_id"], set()).add(n)
        area.setdefault(a["image_id"], {}).setdefault(n, 0.0); area[a["image_id"]][n] += a["area"]
    for im in j["images"]:
        c = per.get(im["id"], set()); ar = area.get(im["id"], {}); tot = sum(ar.values()) or 1.0
        rows.append(dict(file=os.path.join(EV3_ROOT, s, im["file_name"]), classes="|".join(sorted(c)),
                         y=int(any(k.startswith("frozen") for k in c)),
                         frozen_area_frac=round(sum(v for k, v in ar.items() if k.startswith("frozen")) / tot, 3)))
ev = pd.DataFrame(rows)
_N = 32; _C = np.array([[np.cos(np.pi * (2 * x + 1) * u / (2 * _N)) for x in range(_N)] for u in range(_N)])
def phash(img):
    a = np.asarray(img.convert("L").resize((_N, _N), Image.LANCZOS), dtype=np.float64)
    low = (_C @ a @ _C.T)[:8, :8].flatten(); return int("".join("1" if b else "0" for b in low > np.median(low[1:])), 2)
IMGS = []
for f in ev.file:
    im = Image.open(f).convert("RGB"); IMGS.append(im.resize((CFG["IMG_RESIZE"],) * 2, Image.BICUBIC))
ev["phash"] = [phash(Image.open(f)) for f in ev.file]
P = np.array(ev.phash.values, dtype=np.uint64)
D = np.unpackbits((P[:, None] ^ P[None, :]).view(np.uint8).reshape(len(P), len(P), 8), axis=2).sum(2)
par = list(range(len(ev)))
def find(i):
    while par[i] != i: par[i] = par[par[i]]; i = par[i]
    return i
for i, j in zip(*np.nonzero(np.triu(D <= CFG["PHASH_MAX_DIST"], 1))): par[find(i)] = find(j)
ev["group"] = [find(i) for i in range(len(ev))]
fold = np.zeros(len(ev), int)
for k, (_, te) in enumerate(StratifiedGroupKFold(10, shuffle=True, random_state=CFG["EV3_SPLIT_SEED"]).split(ev, ev.y, ev.group)):
    fold[te] = k
ev["split"] = np.where(fold < 3, "test", np.where(fold < 5, "cal", "pool"))
print("near-dup groups with >1 image:", int((ev.group.value_counts() > 1).sum()))
print(ev.pivot_table(index="split", columns="y", values="file", aggfunc="count"))
ev.drop(columns=["phash"]).to_csv(os.path.join(CFG["OUT"], "ev3_split.csv"), index=False)
Y = ev.y.values; SPLIT = ev.split.values

In [ ]:
# 3. WEAPD model: load the v2 checkpoint (or train one), temperature + conformal threshold on the WEAPD val split
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
tf_eval = TV.Compose([TV.CenterCrop(CFG["IMG_SIZE"]), TV.ToTensor(), TV.Normalize(MEAN, STD)])
tf_train = TV.Compose([TV.RandomCrop(CFG["IMG_SIZE"]), TV.RandomHorizontalFlip(), TV.RandomRotation(15),
                       TV.ToTensor(), TV.Normalize(MEAN, STD)])
if SPLIT_CSV:
    wd = pd.read_csv(SPLIT_CSV, dtype={"phash": str})
else:
    assert WEAPD_ROOT, "Neither the v2 output nor the WEAPD dataset was found."
    from sklearn.model_selection import train_test_split
    print("WARNING: v2 split.csv not found -> plain stratified split (not near-duplicate-free).")
    wd = pd.DataFrame([dict(src=os.path.join(WEAPD_ROOT, c, f), label=c.lower()) for c in os.listdir(WEAPD_ROOT)
                       if os.path.isdir(os.path.join(WEAPD_ROOT, c)) for f in os.listdir(os.path.join(WEAPD_ROOT, c))
                       if f.lower().endswith(IMG_EXT)])
    tr, tmp = train_test_split(wd, test_size=0.2, stratify=wd.label, random_state=42)
    va, te = train_test_split(tmp, test_size=0.5, stratify=tmp.label, random_state=42)
    wd["split"] = "train"; wd.loc[va.index, "split"] = "val"; wd.loc[te.index, "split"] = "test"
CLASSES = sorted(wd.label.unique()); C2I = {c: i for i, c in enumerate(CLASSES)}; print(len(CLASSES), "WEAPD classes:", CLASSES)
COLD_IDX = [C2I[c] for c in CFG["COLD_CLASSES"] if c in C2I]
ACTIONS = sorted(set(CFG["ACTION_MAP"].values())); A_OF = np.array([ACTIONS.index(CFG["ACTION_MAP"].get(c, "no_action")) for c in CLASSES])
def weapd_path(r):
    if os.path.exists(str(r.src)): return r.src
    if WEAPD_ROOT:
        p = os.path.join(WEAPD_ROOT, os.path.basename(os.path.dirname(str(r.src))), os.path.basename(str(r.src)))
        if os.path.exists(p): return p
    return None
if WEAPD_ROOT:
    real = {d.lower(): d for d in os.listdir(WEAPD_ROOT)}
    wd["file"] = [weapd_path(r) or os.path.join(WEAPD_ROOT, real.get(r.label, r.label), os.path.basename(str(r.src))) for r in wd.itertuples()]
    wd = wd[[os.path.exists(f) for f in wd.file]].reset_index(drop=True)
    print("WEAPD images found:", wd.split.value_counts().to_dict())

class PILDS(torch.utils.data.Dataset):
    def __init__(self, items, ys, tf, cached=False): self.items, self.ys, self.tf, self.cached = items, ys, tf, cached
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        im = self.items[i] if self.cached else Image.open(self.items[i]).convert("RGB").resize((CFG["IMG_RESIZE"],) * 2, Image.BICUBIC)
        return self.tf(im), self.ys[i]
def loader(items, ys, tf, cached=False, shuffle=False, bs=64):
    return torch.utils.data.DataLoader(PILDS(items, ys, tf, cached), bs, shuffle=shuffle, num_workers=CFG["NUM_WORKERS"],
                                       drop_last=shuffle and len(items) > bs)

@torch.no_grad()
def run(m, dl, feats=False):
    m.eval(); out = []
    for x, _ in dl:
        x = x.to(DEVICE)
        with torch.autocast("cuda", enabled=DEVICE == "cuda"):
            if feats:
                f = lambda z: m.forward_head(m.forward_features(z), pre_logits=True)
                o = (f(x) + f(torch.flip(x, [3]))) / 2
            else:
                o = (m(x) + m(torch.flip(x, [3]))) / 2
        out.append(o.float().cpu())
    return torch.cat(out).numpy()

def train_weapd(b):
    seed_all(0); m = timm.create_model(b, pretrained=True, num_classes=len(CLASSES)).to(DEVICE)
    tr, va = wd[wd.split == "train"], wd[wd.split == "val"]
    dl = loader(tr.file.tolist(), [C2I[l] for l in tr.label], tf_train, shuffle=True, bs=32)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=0.05)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 1e-4, total_steps=CFG["WEAPD_EPOCHS"] * len(dl))
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda"); best, state = -1, None
    for ep in range(CFG["WEAPD_EPOCHS"]):
        m.train()
        for x, y in dl:
            x, y = x.to(DEVICE), y.to(DEVICE); opt.zero_grad()
            with torch.autocast("cuda", enabled=DEVICE == "cuda"): loss = F.cross_entropy(m(x), y, label_smoothing=0.1)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
        pv = run(m, loader(va.file.tolist(), [0] * len(va), tf_eval)).argmax(1)
        f1 = f1_score([C2I[l] for l in va.label], pv, average="macro"); print(f"  {b} ep{ep} val F1 {f1:.4f}")
        if f1 > best: best, state = f1, {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
    m.load_state_dict(state); torch.save(state, os.path.join(CFG["OUT"], f"weapd_{b}.pt")); return m

def load_weapd(b):
    m = timm.create_model(b, pretrained=False, num_classes=len(CLASSES))
    if CKPT[b]:
        sd = torch.load(CKPT[b], map_location="cpu"); sd = sd.get("state_dict", sd)
        print(b, "load:", m.load_state_dict({k.replace("module.", ""): v for k, v in sd.items()}, strict=True)); return m.to(DEVICE)
    assert CFG["TRAIN_IF_MISSING"] and WEAPD_ROOT, f"No checkpoint for {b}"
    print(b, ": checkpoint missing -> training on WEAPD"); return train_weapd(b)

def fit_temperature(logits, y):
    t = torch.zeros(1, requires_grad=True); L = torch.tensor(logits); Yt = torch.tensor(y)
    opt = torch.optim.LBFGS([t], lr=0.1, max_iter=200)
    def cl():
        opt.zero_grad(); l = F.cross_entropy(L / t.exp(), Yt); l.backward(); return l
    opt.step(cl); return float(t.exp())
def softmax(z): z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
def qhat_lac(p_true, alpha):
    n = len(p_true); s = np.sort(1 - p_true); k = int(np.ceil((n + 1) * (1 - alpha))) - 1
    return s[min(k, n - 1)]

In [ ]:
# 4. Zero-shot evaluation of the WEAPD model on EV3
dl_ev = loader(IMGS, list(Y), tf_eval, cached=True)
zs_rows, zs_img = [], []
MODELS = {}
for b in CFG["BACKBONES"]:
    m = load_weapd(b); MODELS[b] = m
    row = dict(backbone=b)
    if WEAPD_ROOT and (wd.split == "val").any():
        va, te = wd[wd.split == "val"], wd[wd.split == "test"]
        lv = run(m, loader(va.file.tolist(), [0] * len(va), tf_eval)); yv = np.array([C2I[l] for l in va.label])
        lt = run(m, loader(te.file.tolist(), [0] * len(te), tf_eval)); yt = np.array([C2I[l] for l in te.label])
        T = fit_temperature(lv, yv); pv, pt = softmax(lv / T), softmax(lt / T)
        row.update(T=round(T, 3), weapd_test_acc=round(float((pt.argmax(1) == yt).mean()), 4))
    else:
        T, pv, yv, pt, yt = 1.0, None, None, None, None
    le = run(m, dl_ev); pe = softmax(le / T)
    cold = pe[:, COLD_IDX].sum(1); top = pe.argmax(1)
    row.update(auroc_cold_score=round(roc_auc_score(Y, cold), 4),
               pct_frost_top_cold_action=round(100 * np.isin(top[Y == 1], COLD_IDX).mean(), 1),
               pct_healthy_top_cold_action=round(100 * np.isin(top[Y == 0], COLD_IDX).mean(), 1),
               mean_maxprob_ev3=round(float(pe.max(1).mean()), 3),
               mean_maxprob_weapd_test=round(float(pt.max(1).mean()), 3) if pt is not None else None)
    for a in CFG["ALPHAS"]:
        if pv is None: break
        q = qhat_lac(pv[np.arange(len(yv)), yv], a)
        S_t = pt >= 1 - q; S_e = pe >= 1 - q
        act_e = np.array([np.isin(np.arange(len(ACTIONS)), np.unique(A_OF[s])) for s in S_e])
        ci = ACTIONS.index("cold_protection")
        row.update({f"a{a}_weapd_test_cov": round(float(S_t[np.arange(len(yt)), yt].mean()), 4),
                    f"a{a}_weapd_test_setsize": round(float(S_t.sum(1).mean()), 3),
                    f"a{a}_ev3_setsize": round(float(S_e.sum(1).mean()), 3),
                    f"a{a}_ev3_empty_pct": round(100 * float((S_e.sum(1) == 0).mean()), 1),
                    f"a{a}_frost_cold_in_set_pct": round(100 * float(act_e[Y == 1, ci].mean()), 1),
                    f"a{a}_healthy_cold_in_set_pct": round(100 * float(act_e[Y == 0, ci].mean()), 1)})
    zs_rows.append(row); print(row)
    for i in range(len(ev)):
        zs_img.append(dict(backbone=b, file=ev.file[i], y=Y[i], top_class=CLASSES[top[i]], top_prob=round(float(pe[i].max()), 3),
                           cold_score=round(float(cold[i]), 3)))
zs = pd.DataFrame(zs_rows); zs.to_csv(os.path.join(CFG["OUT"], "ev3_zero_shot_summary.csv"), index=False)
zi = pd.DataFrame(zs_img); zi.to_csv(os.path.join(CFG["OUT"], "ev3_zero_shot_per_image.csv"), index=False)
print(pd.crosstab([zi.backbone, zi.y], zi.top_class))
pd.crosstab([zi.backbone, zi.y], zi.top_class).to_csv(os.path.join(CFG["OUT"], "ev3_zero_shot_top_class_counts.csv"))
zs.T

In [ ]:
# 5. Features: WEAPD-trained backbone vs the same backbone with ImageNet weights
FEATS = {}
for b in CFG["BACKBONES"]:
    FEATS[(b, "weapd")] = run(MODELS[b], dl_ev, feats=True)
    mi = timm.create_model(b, pretrained=True, num_classes=0).to(DEVICE)
    FEATS[(b, "imagenet")] = run(mi, dl_ev, feats=True); del mi
    print(b, {k[1]: v.shape for k, v in FEATS.items() if k[0] == b})
torch.cuda.empty_cache()

In [ ]:
# 6. Few-shot linear probe (same k-shot samples for both initialisations -> paired comparison)
pool = np.where(SPLIT == "pool")[0]; test = np.where(SPLIT == "test")[0]; cal = np.where(SPLIT == "cal")[0]
def probe(Xtr, ytr, Xte):
    sc = StandardScaler().fit(Xtr); clf = LogisticRegression(C=0.5, max_iter=5000).fit(sc.transform(Xtr), ytr)
    return clf.predict_proba(sc.transform(Xte))[:, 1], (sc, clf)
fs_rows = []
for k in CFG["KSHOTS"] + ["all"]:
    reps = 1 if k == "all" else CFG["REPEATS"]
    for r in range(reps):
        rng = np.random.default_rng(1000 * r + (0 if k == "all" else k))
        idx = pool if k == "all" else np.concatenate([rng.choice(pool[Y[pool] == c], min(k, (Y[pool] == c).sum()), replace=False) for c in (0, 1)])
        for (b, init), X in FEATS.items():
            p, _ = probe(X[idx], Y[idx], X[test])
            fs_rows.append(dict(k=k, rep=r, backbone=b, init=init, auroc=roc_auc_score(Y[test], p),
                                bal_acc=balanced_accuracy_score(Y[test], p >= 0.5), macro_f1=f1_score(Y[test], p >= 0.5, average="macro")))
fs = pd.DataFrame(fs_rows); fs.to_csv(os.path.join(CFG["OUT"], "ev3_fewshot_runs.csv"), index=False)
summ = fs.groupby(["backbone", "init", "k"], sort=False)[["auroc", "bal_acc", "macro_f1"]].agg(["mean", "std"]).round(4)
summ.to_csv(os.path.join(CFG["OUT"], "ev3_fewshot_summary.csv"))
tests = []
for b in CFG["BACKBONES"]:
    for k in CFG["KSHOTS"]:
        a = fs[(fs.backbone == b) & (fs.k == k) & (fs.init == "weapd")].sort_values("rep")
        i = fs[(fs.backbone == b) & (fs.k == k) & (fs.init == "imagenet")].sort_values("rep")
        for met in ["auroc", "bal_acc"]:
            d = a[met].values - i[met].values
            pval = wilcoxon(a[met].values, i[met].values).pvalue if np.any(d != 0) else 1.0
            tests.append(dict(backbone=b, k=k, metric=met, weapd_minus_imagenet=round(float(d.mean()), 4), wilcoxon_p=pval))
pd.DataFrame(tests).to_csv(os.path.join(CFG["OUT"], "ev3_fewshot_wilcoxon.csv"), index=False)
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, len(CFG["BACKBONES"]), figsize=(5 * len(CFG["BACKBONES"]), 3.6), squeeze=False)
for a_, b in zip(ax[0], CFG["BACKBONES"]):
    for init, st in [("weapd", "-o"), ("imagenet", "--s")]:
        g = fs[(fs.backbone == b) & (fs.init == init) & (fs.k != "all")].groupby("k").auroc.agg(["mean", "std"])
        a_.errorbar(g.index.astype(int), g["mean"], g["std"], fmt=st, capsize=3, label=init)
    a_.set_xscale("log"); a_.set_xlabel("labelled images per class"); a_.set_ylabel("test AUROC"); a_.set_title(b.split("_")[0]); a_.legend()
plt.tight_layout(); plt.savefig(os.path.join(CFG["OUT"], "ev3_fewshot_curve.png"), dpi=200)
print(summ); pd.DataFrame(tests)

In [ ]:
# 7. Binary action-level conformal prediction + cost-sensitive threshold (probe trained on the full pool)
cf_rows = []
for (b, init), X in FEATS.items():
    _, (sc, clf) = probe(X[pool], Y[pool], X[pool[:2]])
    pc = clf.predict_proba(sc.transform(X[cal])); pt_ = clf.predict_proba(sc.transform(X[test]))
    for a in CFG["ALPHAS"]:
        q = qhat_lac(pc[np.arange(len(cal)), Y[cal]], a); S = pt_ >= 1 - q
        cov = S[np.arange(len(test)), Y[test]].mean(); size = S.sum(1)
        miss = ((Y[test] == 1) & S[:, 0] & ~S[:, 1]).sum() / max(1, (Y[test] == 1).sum())
        cf_rows.append(dict(backbone=b, init=init, kind="conformal", alpha=a, coverage=round(float(cov), 4),
                            mean_set_size=round(float(size.mean()), 3), singleton_pct=round(100 * float((size == 1).mean()), 1),
                            defer_to_human_pct=round(100 * float((size == 2).mean()), 1), missed_frost_pct=round(100 * float(miss), 2)))
    p1c, p1t = pc[:, 1], pt_[:, 1]
    for r in CFG["COST_RATIOS"]:
        ths = np.linspace(0.01, 0.99, 99)
        costs = [(r * ((p1c < t) & (Y[cal] == 1)).sum() + ((p1c >= t) & (Y[cal] == 0)).sum()) for t in ths]
        t = ths[int(np.argmin(costs))]
        def cost(th): return r * ((p1t < th) & (Y[test] == 1)).sum() + ((p1t >= th) & (Y[test] == 0)).sum()
        cf_rows.append(dict(backbone=b, init=init, kind="cost", cost_ratio=r, threshold=round(float(t), 2),
                            test_cost=int(cost(t)), test_cost_at_0_5=int(cost(0.5)),
                            cost_reduction_pct=round(100 * (1 - cost(t) / max(1, cost(0.5))), 1),
                            missed_frost_pct=round(100 * float(((p1t < t) & (Y[test] == 1)).sum() / (Y[test] == 1).sum()), 2),
                            false_alarm_pct=round(100 * float(((p1t >= t) & (Y[test] == 0)).sum() / (Y[test] == 0).sum()), 2)))
cf = pd.DataFrame(cf_rows); cf.to_csv(os.path.join(CFG["OUT"], "ev3_conformal_cost.csv"), index=False); cf

In [ ]:
# 8. Full fine-tuning on the pool split: WEAPD init vs ImageNet init (early stopping on the cal split)
ft_rows = []
if CFG["FT"] and DEVICE == "cuda":
    items = lambda ix: [IMGS[i] for i in ix]
    for b in CFG["BACKBONES"]:
        for init in ["weapd", "imagenet"]:
            for s in CFG["FT_SEEDS"]:
                seed_all(s); t0 = time.time()
                if init == "weapd":
                    m = timm.create_model(b, pretrained=False, num_classes=len(CLASSES)); m.load_state_dict(MODELS[b].state_dict())
                    m.reset_classifier(2)
                else:
                    m = timm.create_model(b, pretrained=True, num_classes=2)
                m = m.to(DEVICE)
                dl = loader(items(pool), list(Y[pool]), tf_train, cached=True, shuffle=True, bs=CFG["FT_BATCH"])
                opt = torch.optim.AdamW(m.parameters(), lr=CFG["FT_LR"], weight_decay=0.05)
                sch = torch.optim.lr_scheduler.OneCycleLR(opt, CFG["FT_LR"], total_steps=CFG["FT_EPOCHS"] * len(dl))
                scaler = torch.cuda.amp.GradScaler(); best, state = -1, None
                for ep in range(CFG["FT_EPOCHS"]):
                    m.train()
                    for x, y in dl:
                        x, y = x.to(DEVICE), y.to(DEVICE); opt.zero_grad()
                        with torch.autocast("cuda"): loss = F.cross_entropy(m(x), y)
                        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
                    pcal = softmax(run(m, loader(items(cal), list(Y[cal]), tf_eval, cached=True)))[:, 1]
                    au = roc_auc_score(Y[cal], pcal)
                    if au > best: best, state = au, {k: v.detach().clone() for k, v in m.state_dict().items()}
                m.load_state_dict(state)
                pte = softmax(run(m, loader(items(test), list(Y[test]), tf_eval, cached=True)))[:, 1]
                ft_rows.append(dict(backbone=b, init=init, seed=s, cal_auroc=round(best, 4), test_auroc=roc_auc_score(Y[test], pte),
                                    test_bal_acc=balanced_accuracy_score(Y[test], pte >= 0.5),
                                    test_macro_f1=f1_score(Y[test], pte >= 0.5, average="macro"), minutes=round((time.time() - t0) / 60, 1)))
                print(ft_rows[-1]); del m; torch.cuda.empty_cache()
ft = pd.DataFrame(ft_rows); ft.to_csv(os.path.join(CFG["OUT"], "ev3_finetune_runs.csv"), index=False)
if len(ft):
    fts = ft.groupby(["backbone", "init"])[["test_auroc", "test_bal_acc", "test_macro_f1"]].agg(["mean", "std"]).round(4)
    fts.to_csv(os.path.join(CFG["OUT"], "ev3_finetune_summary.csv")); print(fts)

In [ ]:
# 9. Contact sheet of zero-shot predictions (first backbone) and small zip for download
b0 = CFG["BACKBONES"][0]; z0 = zi[zi.backbone == b0].reset_index(drop=True)
pick = list(z0[z0.y == 1].sample(8, random_state=0).index) + list(z0[z0.y == 0].sample(8, random_state=0).index)
W = 200; sheet = Image.new("RGB", (4 * W, 4 * (W + 18)), "white"); dr = ImageDraw.Draw(sheet)
for n, i in enumerate(pick):
    x0, y0 = (n % 4) * W, (n // 4) * (W + 18); sheet.paste(IMGS[i].resize((W, W)), (x0, y0))
    dr.text((x0 + 2, y0 + W + 2), f"{'FROST' if Y[i] else 'healthy'} -> {z0.top_class[i]} {z0.top_prob[i]:.2f} cold={z0.cold_score[i]:.2f}",
            fill=(200, 0, 0) if Y[i] else (0, 100, 0))
sheet.save(os.path.join(CFG["OUT"], "ev3_zero_shot_examples.jpg"), quality=88)
import zipfile
zp = "/kaggle/working/ev3_results_small.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(CFG["OUT"]):
        if not f.endswith(".pt"): z.write(os.path.join(CFG["OUT"], f), f)
print(zp, "%.1f MB" % (os.path.getsize(zp) / 1e6), sorted(os.listdir(CFG["OUT"])))